## 1. Setup and Configuration

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

# =============================================================================
# DIRECTORY PATHS
# =============================================================================
CODE_DIR      = Path(r"C:\Users\willi\.vscode\Github\ml-from-crowd")
DATA_DIR      = Path(r"E:\Research_data\Stocktwits\dataset\v1\data\csv")
INPUT_FOLDER  = DATA_DIR / "merged_with_crsp_mlcrowd"
OUTPUT_FOLDER = DATA_DIR / "features_mlcrowd"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE   = OUTPUT_FOLDER / "features_07_user_influence_accuracy.pkl"

# =============================================================================
# PARAMETERS
# =============================================================================
SKILL_HORIZONS = [21, 63]   # trading-day lags for track-record outcome window
SKILL_RET_COL  = "ar_capm_21"   # forward abnormal return used as call outcome
SHRINK         = 20.0            # Bayesian pseudo-count (strength of no-skill prior)

print(f"Input : {INPUT_FOLDER}")
print(f"Output: {OUTPUT_FILE}")
print(f"Skill return column: {SKILL_RET_COL}  Horizons: {SKILL_HORIZONS} td")

## 2. Load Trading Day Calendar

In [ ]:
import pandas_datareader.data as pdr

print("Fetching Fama-French trading days...")
ff   = pdr.DataReader("F-F_Research_Data_Factors_daily", "famafrench", start="2009-01-01")[0]
trading_days = pd.DatetimeIndex(ff.index).normalize().unique().sort_values()
td_idx       = {d: i for i, d in enumerate(trading_days)}
td_sorted    = sorted(td_idx, key=lambda d: td_idx[d])
print(f"Trading calendar: {len(trading_days):,} days")

## 3. Load Sample Data for Development

In [ ]:
files = sorted([f for f in os.listdir(INPUT_FOLDER) if f.endswith(".csv")])
print(f"Files found: {len(files)}")

df_sample = pd.read_csv(INPUT_FOLDER / files[-1])
df_sample["date"]       = pd.to_datetime(df_sample["date"])
df_sample["created_at"] = pd.to_datetime(df_sample["created_at"])
print(f"Sample shape: {df_sample.shape}")
print(f"Date range: {df_sample['date'].min().date()} to {df_sample['date'].max().date()}")
print(f"AR columns available: {[c for c in df_sample.columns if c.startswith('ar_')]}")
display(df_sample[["user_id","date","symbol","sentiment","ar_capm_21"]].head(5))

## 4. Core Functions

### 4a. Build per-user expanding skill table (no look-ahead)

In [ ]:
def build_user_skill_table(df, lag_td=21, ret_col="ar_capm_21", shrink=SHRINK):
    """
    Compute each user's shrunk directional hit-rate (skill score) from their
    historical calls, with a lag_td-trading-day lag to prevent look-ahead.

    For a call made on date d with known forward return ret_col:
      - The outcome is available at date d + lag_td.
      - This call updates the user's skill score available at d + lag_td + 1.
      - Any message on date t uses the skill score from calls whose
        availability_date <= t, i.e. calls made on or before t - lag_td - 1.

    Shrinkage formula (Bayesian pseudo-count):
        shrunk_hit_rate = (cum_hits + shrink * 0.5) / (cum_n + shrink)
        skill_score     = 2 * (shrunk_hit_rate - 0.5)   # centred at 0

    A user with no history has skill_score = 0.0 (no-skill prior).

    Parameters
    ----------
    df       : DataFrame for current year (or all years combined)
    lag_td   : int, trading-day look-back lag
    ret_col  : str, forward-return column (e.g. 'ar_capm_21')
    shrink   : float, Bayesian pseudo-count for shrinkage

    Returns
    -------
    skill_timeline : pd.DataFrame
        Columns: user_id, date (= availability_date), skill_score
        Sorted by (user_id, date).
        Merge into message-level df via attach_user_skill().
    """
    tagged = df[df["sentiment"].isin(["Bullish", "Bearish"])].copy()
    tagged["direction"] = tagged["sentiment"].map({"Bullish": 1, "Bearish": -1})
    tagged["hit"]       = (
        tagged["direction"] * tagged[ret_col] > 0
    ).astype(float)
    tagged = tagged.sort_values(
        ["user_id", "date", "created_at"]
    ).reset_index(drop=True)

    # Expanding cumulative stats per user (shift(1) ? stats BEFORE current call)
    g = tagged.groupby("user_id")
    tagged["cum_n"]    = g.cumcount()                         # 0-indexed
    tagged["cum_hits"] = g["hit"].cumsum().shift(1).fillna(0)

    tagged["shrunk_hr"]   = (
        (tagged["cum_hits"] + shrink * 0.5) / (tagged["cum_n"] + shrink)
    )
    tagged["skill_score"] = 2 * (tagged["shrunk_hr"] - 0.5)

    # Availability date: call made on date d ? skill usable at d + lag_td
    max_td = max(td_idx.values())
    def _avail(d):
        i = td_idx.get(d)
        if i is None:
            return pd.NaT
        fi = i + lag_td
        return td_sorted[fi] if fi <= max_td else pd.NaT

    tagged["avail_date"] = tagged["date"].apply(_avail)
    tl = (
        tagged.dropna(subset=["avail_date"])
        [["user_id", "avail_date", "skill_score"]]
        .rename(columns={"avail_date": "date"})
        .sort_values(["user_id", "date"])
        .reset_index(drop=True)
    )
    return tl


def attach_user_skill(df, skill_timeline, default=0.0):
    """
    Attach the most-recently-available skill score to each message row.

    Uses forward-fill within user groups: for each row at date t, takes the
    last skill_score with availability_date <= t (i.e. derived from calls made
    strictly before t - lag_td, so no look-ahead).

    Messages before any skill update receive skill_score = default (0.0 = no-skill).
    """
    sk = skill_timeline.assign(is_sk=True)
    mg = df[["user_id", "date"]].assign(skill_score=np.nan, is_sk=False)
    combo = pd.concat(
        [sk[["user_id", "date", "skill_score", "is_sk"]],
         mg[["user_id", "date", "skill_score", "is_sk"]]]
    ).sort_values(["user_id", "date", "is_sk"]).reset_index(drop=True)

    combo["user_skill"] = (
        combo.groupby("user_id")["skill_score"].ffill().fillna(default)
    )
    msgs_only = combo[~combo["is_sk"]].copy()
    return df.assign(user_skill=msgs_only["user_skill"].values)

### 4b. Firm-day skill-weighted sentiment features

In [ ]:
def calc_skill_features(df, skill_tl, lag_label="21"):
    """
    Aggregate per-user skill scores to firm-day features.

    Features
    --------
    skill_wtd_net_sent_{lag}   Skill-weighted net sentiment; analogous to
                                net_sentiment but users are weighted by their
                                historical accuracy rather than raw follower count.
    avg_poster_skill_{lag}     Mean skill score among all tagged posters today.
    skilled_bull_ratio_{lag}   Fraction of skilled (skill>0) posters who are Bullish.
    n_skilled_posters_{lag}    Count of users with skill_score > 0.
    skill_dispersion_{lag}     Std dev of skill scores among today's posters
                                (high = heterogeneous crowd; low = homogeneous).
    """
    df_sk = attach_user_skill(df, skill_tl, default=0.0)
    tagged = df_sk[df_sk["sentiment"].isin(["Bullish", "Bearish"])].copy()
    tagged["direction"] = tagged["sentiment"].map({"Bullish": 1, "Bearish": -1})

    # Skill weight: shift so 0-skill user contributes 0.5 (neutral but not zero)
    tagged["w"] = tagged["user_skill"].clip(lower=0) + 0.5

    def _agg(g):
        bull_w  = (g["w"] * (g["direction"] == 1)).sum()
        bear_w  = (g["w"] * (g["direction"] == -1)).sum()
        tot_w   = bull_w + bear_w
        skilled = g["user_skill"] > 0
        return pd.Series({
            f"skill_wtd_net_sent_{lag_label}":  (bull_w - bear_w) / tot_w if tot_w else np.nan,
            f"avg_poster_skill_{lag_label}":     g["user_skill"].mean(),
            f"skilled_bull_ratio_{lag_label}":  (skilled & (g["direction"]==1)).sum() /
                                                  skilled.sum() if skilled.sum() > 0 else np.nan,
            f"n_skilled_posters_{lag_label}":    int(skilled.sum()),
            f"skill_dispersion_{lag_label}":     g["user_skill"].std(),
        })

    result = (
        tagged.groupby(["symbol", "date"], group_keys=False)
        .apply(_agg)
        .reset_index()
    )
    return result


def calc_all_skill_features(df, lag_td_list=SKILL_HORIZONS,
                             ret_col=SKILL_RET_COL, shrink=SHRINK):
    """Build skill features for each lag horizon and merge them."""
    result = None
    for lag in lag_td_list:
        tl  = build_user_skill_table(df, lag_td=lag, ret_col=ret_col, shrink=shrink)
        sf  = calc_skill_features(df, tl, lag_label=str(lag))
        result = sf if result is None else result.merge(
            sf, on=["symbol", "date"], how="outer"
        )
    return result

## 5. Test on Sample Data

In [ ]:
print("Building skill table for sample year (lag=21 td)...")
tl_sample = build_user_skill_table(df_sample, lag_td=21, ret_col=SKILL_RET_COL)
print(f"Skill timeline rows: {len(tl_sample):,}")
print(f"Unique users with skill history: {tl_sample['user_id'].nunique():,}")
print("\nSample skill timeline (first 10 rows):")
display(tl_sample.head(10))

print("\nBuilding skill-weighted sentiment features...")
sf_sample = calc_all_skill_features(df_sample)
print(f"Skill feature shape: {sf_sample.shape}")
print(f"Columns: {list(sf_sample.columns)}")
display(sf_sample.head(10))

# Quick check: skill_wtd_net_sent should be in [-1, 1]
for col in [c for c in sf_sample.columns if "wtd_net_sent" in c]:
    mn, mx = sf_sample[col].min(), sf_sample[col].max()
    print(f"  {col}: min={mn:.3f}  max={mx:.3f}  (should be in [-1,1])")

## 6. Visualize Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

cols_to_plot = [c for c in sf_sample.columns if c not in ("symbol","date")]

for ax, col in zip(axes.flatten(), cols_to_plot[:6]):
    vals = sf_sample[col].dropna()
    ax.hist(vals, bins=40, edgecolor="black", alpha=0.7)
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.axvline(vals.mean(), color="red", ls="--",
               label=f"mean={vals.mean():.3f}")
    ax.legend(fontsize=8)

plt.suptitle("User Influence & Accuracy Features ? Sample Year", fontsize=13)
plt.tight_layout(); plt.show()

# Correlation with raw net_sentiment from features_01
if "net_sentiment" in df_sample.columns:
    net_sent = (
        df_sample[df_sample["sentiment"].isin(["Bullish","Bearish"])]
        .groupby(["symbol","date"]).apply(
            lambda g: (g["sentiment"]=="Bullish").sum() /
                      max(len(g), 1) * 2 - 1
        ).rename("net_sentiment").reset_index()
    )
    compare = sf_sample.merge(net_sent, on=["symbol","date"], how="inner")
    for lag in SKILL_HORIZONS:
        col = f"skill_wtd_net_sent_{lag}"
        if col in compare:
            r = compare[[col, "net_sentiment"]].dropna().corr().iloc[0,1]
            print(f"Corr(skill_wtd_net_sent_{lag}, raw_net_sentiment) = {r:.4f}")

## 7. Feature Statistics & Validation

In [ ]:
feature_cols = [c for c in sf_sample.columns if c not in ("symbol","date")]
print(f"Feature columns ({len(feature_cols)}):")
print(feature_cols)
print()
display(sf_sample[feature_cols].describe().round(4))

# Checks
for lag in SKILL_HORIZONS:
    col = f"skill_wtd_net_sent_{lag}"
    if col in sf_sample:
        assert sf_sample[col].between(-1.01,1.01,inclusive="both").dropna().all()
        print(f"  {col}: bounds OK")
    col2 = f"avg_poster_skill_{lag}"
    if col2 in sf_sample:
        print(f"  {col2}: mean={sf_sample[col2].mean():.4f}  "
              f"(should be near 0 for first year with no history)")
print("\nAll validation checks passed.")

## 8. Process All Years

In [ ]:
# Strategy: for each year, build skill table using ALL data up to t-lag_td.
# We accumulate calls across years in a growing buffer so skill for year Y+1
# benefits from years 1..Y. This is the walk-forward, no-lookahead approach.
#
# Memory note: the running calls_buffer grows up to ~full dataset but only
# the columns needed for skill (user_id, date, sentiment, ar_*) are kept.

all_features = []
processing_stats = []

# Accumulate previous year data for skill computation
calls_buffer = pd.DataFrame(columns=["user_id","date","created_at",
                                      "sentiment", SKILL_RET_COL])

print(f"{'='*60}")
print("Processing all years (chronological order)...")
print(f"{'='*60}\n")

for file in tqdm(files, desc="Processing years"):
    try:
        year = file.split("_")[-1].replace(".csv","")
        df_year = pd.read_csv(INPUT_FOLDER / file)
        df_year["date"]       = pd.to_datetime(df_year["date"])
        df_year["created_at"] = pd.to_datetime(df_year["created_at"])

        # Build skill from ALL data up to (but not including) current year
        # The lag_td offset inside build_user_skill_table ensures no lookahead
        # within the combined buffer.
        combined_for_skill = pd.concat(
            [calls_buffer, df_year[calls_buffer.columns]], ignore_index=True
        ) if len(calls_buffer) > 0 else df_year[calls_buffer.columns].copy()

        sf_year = calc_all_skill_features(
            combined_for_skill,
            lag_td_list=SKILL_HORIZONS,
            ret_col=SKILL_RET_COL,
        )
        # Filter to current year only (skill features for this year's messages)
        cur_dates = df_year["date"].unique()
        sf_year = sf_year[sf_year["date"].isin(cur_dates)].copy()

        all_features.append(sf_year)
        processing_stats.append({
            "year": year,
            "input_rows":    len(df_year),
            "feature_rows":  len(sf_year),
            "unique_symbols": sf_year["symbol"].nunique(),
        })

        # Grow the buffer with current year's call data
        calls_buffer = pd.concat(
            [calls_buffer, df_year[calls_buffer.columns].copy()],
            ignore_index=True
        )

    except Exception as e:
        print(f"  ERROR in {file}: {e}")
        import traceback; traceback.print_exc()

features_all = pd.concat(all_features, ignore_index=True)
features_all = features_all.sort_values(["symbol","date"]).reset_index(drop=True)

print(f"\nDone!  Total feature rows: {len(features_all):,}")
print(f"Unique symbols: {features_all['symbol'].nunique():,}")
print(f"Date range: {features_all['date'].min()} to {features_all['date'].max()}")
display(pd.DataFrame(processing_stats))

## 9. Final Data Inspection

In [ ]:
print(f"Shape: {features_all.shape}")
print(f"\nColumns: {list(features_all.columns)}")
print(f"\nMemory: {features_all.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"\nNull counts:")
print(features_all.isnull().sum())
print("\nSummary statistics:")
display(features_all.describe().round(4))

## 10. Save to Pickle

In [ ]:
print(f"Saving to: {OUTPUT_FILE}")
features_all.to_pickle(OUTPUT_FILE)

verify = pd.read_pickle(OUTPUT_FILE)
assert verify.shape == features_all.shape
file_mb = OUTPUT_FILE.stat().st_size / 1024**2
print(f"Saved and verified. File size: {file_mb:.1f} MB")

## Summary

**Rationale**

Social media signals vary enormously in predictive value across users.  A Goldman
Sachs analyst's Bullish post carries more signal than a first-time retail poster's
identical post.  Without follower/employer metadata in this pipeline, we cannot use
reach- or employer-based weighting ? but we *can* score users by their actual
historical accuracy, which is arguably the most valid signal of all.

**Methodology**

1. For each user and each tagged post (direction = +1 Bullish / -1 Bearish):
   - The *realized outcome* is the forward abnormal return (`ar_capm_21`) already
     present in the dataset (the same return used as the ML target elsewhere).
   - A call is a "hit" if `direction ? fwd_return > 0`.
2. Expanding cumulative hit-rate per user is shrunk toward 0.5 (no skill) using a
   Bayesian pseudo-count of `SHRINK = 20`.  A new user starts at 0.5; a user needs
   ~40 tracked calls to substantially deviate.
3. Skill is made available with a `lag_td`-trading-day lag so that the realized
   return for call at t is NOT used before it could be known (no look-ahead).
4. At the firm-day level, users' messages are weighted by their skill score to
   produce `skill_wtd_net_sent_H` ? distinct from the raw `net_sentiment` in
   features_01 because high-accuracy users dominate.

**Features produced**

| Column | Description |
|--------|-------------|
| `skill_wtd_net_sent_H` | Skill-weighted net sentiment (H-td lag) |
| `avg_poster_skill_H` | Mean skill score among today's tagged posters |
| `skilled_bull_ratio_H` | Fraction of *skilled* posters who are Bullish |
| `n_skilled_posters_H` | Count of users with skill > 0 today |
| `skill_dispersion_H` | Std dev of skill scores (crowd quality heterogeneity) |

H ? {21, 63} trading days.  Output: `features_07_user_influence_accuracy.pkl`

**Look-ahead safety note**: The `ar_capm_21` column represents the 21-day forward
abnormal return from each message date.  When used as input to `build_user_skill_table`
with `lag_td=21`, a call's contribution to the skill table first appears at
`call_date + 21 td`, so it can only influence messages on dates ? `call_date + 22 td`.
No message's label is leaked into its own feature row.
